In [3]:
# !pip install pandas numpy scanpy scrublet

In [1]:
import sys
sys.path.insert(0, '/home/ubuntu/.local/lib/python3.12/site-packages')

In [2]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 
import glob

In [3]:
import anndata as ad
ad.settings.allow_write_nullable_strings = True

In [6]:
!curl -L -o dropbox_folder.zip "https://www.dropbox.com/scl/fo/ay3uu1f821zvh87r84rim/ANGu9Jlt8ZS0OxXTCvd2WVY?rlkey=4j0ja6zwc1ige61ru4053zd43&dl=1"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     17      0  0:00:01 --:--:--  0:00:01    17
100 18.2M  100 18.2M    0     0  5328k      0  0:00:03  0:00:03 --:--:-- 13.7M


In [7]:
!curl -L -o dropbox_folder_data.zip "https://www.dropbox.com/scl/fo/udwlmgj3iryvfqfcfzoqb/AGSjWeUV6c21jwtQfgnmWa8?rlkey=5e28p9t0iu0dtyy3ff872xflo&dl=1"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     16      0  0:00:01  0:00:01 --:--:--    16
100 4389M  100 4389M    0     0  20.5M      0  0:03:33  0:03:33 --:--:-- 32.1M


In [9]:
!unzip "*.zip"

Archive:  dropbox_folder_data.zip
mapname:  conversion of  failed
 extracting: Data_Wu2021_Breast.tar.gz  
 extracting: Data_Kim2018_Breast.tar.gz  
 extracting: Data_Gao2021_Breast.tar.gz  
 extracting: Data_Pal2021_Breast.tar.gz  
 extracting: Data_Qian2020_Breast.tar.gz  
 extracting: Data_Chung2017_Breast.tar.gz  
 extracting: Data_Savas2018_Breast.tar.gz  
 extracting: Data_Azizi2018_Breast.tar.gz  
 extracting: Data_Gulati2020_Breast.tar.gz  
 extracting: Data_Bassez2021_Breast.tar.gz  
 extracting: Data_Karaayvas2018_Breast.tar.gz  
 extracting: Data_Griffiths2021_Breast.tar.gz  

Archive:  dropbox_folder.zip
mapname:  conversion of  failed
 extracting: Meta-data_Wu2021_Breast.tar.gz  
 extracting: Meta-data_Kim2018_Breast.tar.gz  
 extracting: Meta-data_Gao2021_Breast.tar.gz  
 extracting: Meta-data_Pal2021_Breast.tar.gz  
 extracting: Meta-data_Qian2020_Breast.tar.gz  
 extracting: Meta-data_Chung2017_Breast.tar.gz  
 extracting: Meta-data_Savas2018_Breast.tar.gz  
 extracting

In [11]:
!for f in *.tar.gz; do tar -xzvf "$f"; done

Data_Azizi2018_Breast/
Data_Azizi2018_Breast/InDrop/
Data_Azizi2018_Breast/InDrop/Cells.csv
Data_Azizi2018_Breast/InDrop/Genes.txt
Data_Azizi2018_Breast/InDrop/Exp_data_UMIcounts.mtx
Data_Azizi2018_Breast/Samples.csv
Data_Azizi2018_Breast/10x/
Data_Azizi2018_Breast/10x/Cells.csv
Data_Azizi2018_Breast/10x/Genes.txt
Data_Azizi2018_Breast/10x/Exp_data_UMIcounts.mtx
Data_Bassez2021_Breast/
Data_Bassez2021_Breast/Samples.csv
Data_Bassez2021_Breast/Cells.csv
Data_Bassez2021_Breast/Genes.txt
Data_Bassez2021_Breast/Exp_data_UMIcounts.mtx
Data_Chung2017_Breast/
Data_Chung2017_Breast/Exp_data_TPM.mtx
Data_Chung2017_Breast/Samples.csv
Data_Chung2017_Breast/Cells.csv
Data_Chung2017_Breast/Genes.txt
Data_Gao2021_Breast/
Data_Gao2021_Breast/Breast/
Data_Gao2021_Breast/Breast/Cells.csv
Data_Gao2021_Breast/Breast/Genes.txt
Data_Gao2021_Breast/Breast/Exp_data_UMIcounts.mtx
Data_Gao2021_Breast/Samples.csv
Data_Gao2021_Breast/Breast_and_thyroid/
Data_Gao2021_Breast/Breast_and_thyroid/Cells_breast_and_thy

In [4]:
study_files = {
    # Azizi2018 – two platforms
    "Azizi2018_InDrop": {
        "matrix": "Data_Azizi2018_Breast/InDrop/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Azizi2018_Breast/InDrop/Genes.txt",
        "cells":  "Data_Azizi2018_Breast/InDrop/Cells.csv",
        "samples": "Data_Azizi2018_Breast/Samples.csv",
        "units": "UMI"
    },
    "Azizi2018_10x": {
        "matrix": "Data_Azizi2018_Breast/10x/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Azizi2018_Breast/10x/Genes.txt",
        "cells":  "Data_Azizi2018_Breast/10x/Cells.csv",
        "samples": "Data_Azizi2018_Breast/Samples.csv",
        "units": "UMI"
    },
    # Bassez2021 – single platform
    "Bassez2021": {
        "matrix": "Data_Bassez2021_Breast/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Bassez2021_Breast/Genes.txt",
        "cells":  "Data_Bassez2021_Breast/Cells.csv",
        "samples": "Data_Bassez2021_Breast/Samples.csv",
        "units": "UMI"
    },
    # Chung2017 – TPM
    "Chung2017": {
        "matrix": "Data_Chung2017_Breast/Exp_data_TPM.mtx",
        "genes":  "Data_Chung2017_Breast/Genes.txt",
        "cells":  "Data_Chung2017_Breast/Cells.csv",
        "samples": "Data_Chung2017_Breast/Samples.csv",
        "units": "TPM"
    },
    # Gao2021 – two sub‑studies
    "Gao2021_Breast": {
        "matrix": "Data_Gao2021_Breast/Breast/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Gao2021_Breast/Breast/Genes.txt",
        "cells":  "Data_Gao2021_Breast/Breast/Cells.csv",
        "samples": "Data_Gao2021_Breast/Samples.csv",
        "units": "UMI"
    },
    "Gao2021_Breast_and_thyroid": {
        "matrix": "Data_Gao2021_Breast/Breast_and_thyroid/exp_data_UMIcountscells_breast_and_thyroid.mtx",
        "genes":  "Data_Gao2021_Breast/Breast_and_thyroid/genes_breast_and_thyroid.txt",
        "cells":  "Data_Gao2021_Breast/Breast_and_thyroid/Cells_breast_and_thyroid.csv",
        "samples": "Data_Gao2021_Breast/Samples.csv",
        "units": "UMI"
    },
    # Griffiths2021 – two platforms
    "Griffiths2021_iCell8": {
        "matrix": "Data_Griffiths2021_Breast/iCell8/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Griffiths2021_Breast/iCell8/Genes.txt",
        "cells":  "Data_Griffiths2021_Breast/iCell8/Cells.csv",
        "samples": "Data_Griffiths2021_Breast/Samples.csv",
        "units": "UMI"
    },
    "Griffiths2021_10X": {
        "matrix": "Data_Griffiths2021_Breast/10X/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Griffiths2021_Breast/10X/Genes.txt",
        "cells":  "Data_Griffiths2021_Breast/10X/Cells.csv",
        "samples": "Data_Griffiths2021_Breast/Samples.csv",
        "units": "UMI"
    },
    # Gulati2020 – TPM
    "Gulati2020": {
        "matrix": "Data_Gulati2020_Breast/Exp_data_TPM.mtx",
        "genes":  "Data_Gulati2020_Breast/Genes.txt",
        "cells":  "Data_Gulati2020_Breast/Cells.csv",
        "samples": "Data_Gulati2020_Breast/Samples.csv",
        "units": "TPM"
    },
    # Karaayvas2018 – TPM
    "Karaayvas2018": {
        "matrix": "Data_Karaayvas2018_Breast/Exp_data_TPM.mtx",
        "genes":  "Data_Karaayvas2018_Breast/Genes.txt",
        "cells":  "Data_Karaayvas2018_Breast/Cells.csv",
        "samples": "Data_Karaayvas2018_Breast/Samples.csv",
        "units": "TPM"
    },
    # Kim2018 – TPM
    "Kim2018": {
        "matrix": "Data_Kim2018_Breast/Exp_data_TPM.mtx",
        "genes":  "Data_Kim2018_Breast/Genes.txt",
        "cells":  "Data_Kim2018_Breast/Cells.csv",
        "samples": "Data_Kim2018_Breast/Samples.csv",
        "units": "TPM"
    },
    # Pal2021 – five groups sharing the same Genes.txt in parent folder
    "Pal2021_Group1": {
        "matrix": "Data_Pal2021_Breast/Group1/Exp_data_UMIcounts1.mtx",
        "genes":  "Data_Pal2021_Breast/Genes.txt",
        "cells":  "Data_Pal2021_Breast/Group1/Cells1.csv",
        "samples": "Data_Pal2021_Breast/Samples.csv",
        "units": "UMI"
    },
    "Pal2021_Group2": {
        "matrix": "Data_Pal2021_Breast/Group2/Exp_data_UMIcounts2.mtx",
        "genes":  "Data_Pal2021_Breast/Genes.txt",
        "cells":  "Data_Pal2021_Breast/Group2/Cells2.csv",
        "samples": "Data_Pal2021_Breast/Samples.csv",
        "units": "UMI"
    },
    "Pal2021_Group3": {
        "matrix": "Data_Pal2021_Breast/Group3/Exp_data_UMIcounts3.mtx",
        "genes":  "Data_Pal2021_Breast/Genes.txt",
        "cells":  "Data_Pal2021_Breast/Group3/Cells3.csv",
        "samples": "Data_Pal2021_Breast/Samples.csv",
        "units": "UMI"
    },
    "Pal2021_Group4": {
        "matrix": "Data_Pal2021_Breast/Group4/Exp_data_UMIcounts4.mtx",
        "genes":  "Data_Pal2021_Breast/Genes.txt",
        "cells":  "Data_Pal2021_Breast/Group4/Cells4.csv",
        "samples": "Data_Pal2021_Breast/Samples.csv",
        "units": "UMI"
    },
    "Pal2021_Group5": {
        "matrix": "Data_Pal2021_Breast/Group5/Exp_data_UMIcounts5.mtx",
        "genes":  "Data_Pal2021_Breast/Genes.txt",
        "cells":  "Data_Pal2021_Breast/Group5/Cells5.csv",
        "samples": "Data_Pal2021_Breast/Samples.csv",
        "units": "UMI"
    },
    # Qian2020
    "Qian2020": {
        "matrix": "Data_Qian2020_Breast/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Qian2020_Breast/Genes.txt",
        "cells":  "Data_Qian2020_Breast/Cells.csv",
        "samples": "Data_Qian2020_Breast/Samples.csv",
        "units": "UMI"
    },
    # Savas2018
    "Savas2018": {
        "matrix": "Data_Savas2018_Breast/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Savas2018_Breast/Genes.txt",
        "cells":  "Data_Savas2018_Breast/Cells.csv",
        "samples": "Data_Savas2018_Breast/Samples.csv",
        "units": "UMI"
    },
    # Wu2021
    "Wu2021": {
        "matrix": "Data_Wu2021_Breast/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Wu2021_Breast/Genes.txt",
        "cells":  "Data_Wu2021_Breast/Cells.csv",
        "samples": "Data_Wu2021_Breast/Samples.csv",
        "units": "UMI"
    }
}

In [13]:
for study_name, paths in study_files.items():
    adata = sc.read_mtx(paths['matrix']).T
    cells = pd.read_csv(paths['cells'], index_col=0)
    print(f"{study_name}: {adata.n_obs} cells in matrix, {len(cells)} cells in metadata")

Azizi2018_InDrop: 47016 cells in matrix, 47016 cells in metadata
Azizi2018_10x: 28013 cells in matrix, 28013 cells in metadata
Bassez2021: 226635 cells in matrix, 226635 cells in metadata
Chung2017: 515 cells in matrix, 515 cells in metadata
Gao2021_Breast: 4143 cells in matrix, 4143 cells in metadata
Gao2021_Breast_and_thyroid: 25849 cells in matrix, 25849 cells in metadata
Griffiths2021_iCell8: 804 cells in matrix, 804 cells in metadata
Griffiths2021_10X: 110568 cells in matrix, 110568 cells in metadata
Gulati2020: 1902 cells in matrix, 1902 cells in metadata
Karaayvas2018: 1534 cells in matrix, 1534 cells in metadata
Kim2018: 2472 cells in matrix, 2472 cells in metadata
Pal2021_Group1: 62887 cells in matrix, 62887 cells in metadata
Pal2021_Group2: 44225 cells in matrix, 44225 cells in metadata
Pal2021_Group3: 64201 cells in matrix, 64201 cells in metadata
Pal2021_Group4: 65129 cells in matrix, 65129 cells in metadata
Pal2021_Group5: 68782 cells in matrix, 68782 cells in metadata
Qia

/tmp/ipykernel_340/3156358137.py:3: DtypeWarning: Columns (0: subtype_IHC) have mixed types. Specify dtype option on import or set low_memory=False.
  cells = pd.read_csv(paths['cells'], index_col=0)


In [5]:
study_name = 'Azizi2018_InDrop'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 47,016 cells × 14,869 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 45,579 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.77
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 1.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 90.2 seconds
After doublet removal: 45,579 cells
After MT filter: 44,555 cells
Normalised (UMI) – max value 12.73


In [6]:
study_name = 'Azizi2018_10x'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 28,013 cells × 12,908 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 28,003 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.73
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 5.1%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 51.1 seconds
After doublet removal: 28,003 cells
After MT filter: 27,933 cells
Normalised (UMI) – max value 13.68


In [ ]:
study_name = 'Bassez2021'
# no vals in col to distinguish cancer type

Loaded: 226,635 cells × 22,567 genes
Flagged 0 cells
   Added cancer_type column. Unique values: <StringArray>
['Breast Cancer']
Length: 1, dtype: str


In [7]:
study_name = 'Chung2017'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

/tmp/ipykernel_488/1491167169.py:9: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['SNORD113-1', 'SNORD113-2', 'SNORD113-3', 'SNORD113-4', 'SNORD113-5']
  adata.var_names_make_unique()


Loaded: 515 cells × 57,915 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 275 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.21
Detected doublet rate = 0.7%
Estimated detectable doublet fraction = 9.1%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 8.0%
Elapsed time: 0.6 seconds
After doublet removal: 273 cells
After MT filter: 155 cells
Normalised (TPM) – max value 13.24


In [8]:
study_name = 'Gao2021_Breast'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 4,143 cells × 33,694 genes
   cell_type: 251 NaN
   cell_type: 251 suspicious
Flagged 251 cells
After metadata drop: 3,892 cells
   Added cancer_type column. Unique values: ['Premalignant' 'Breast Cancer']
After gene filter: 3,365 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.46
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 34.4%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.2%
Elapsed time: 5.8 seconds
After doublet removal: 3,363 cells
After MT filter: 3,214 cells
Normalised (UMI) – max value 13.38


In [9]:
study_name = 'Gao2021_Breast_and_thyroid'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 25,849 cells × 33,538 genes
   cell_type: 4737 NaN
   cell_type: 4737 suspicious
Flagged 4737 cells
After metadata drop: 21,112 cells
   Added cancer_type column. Unique values: ['Thyroid Cancer' 'Breast Cancer']
After gene filter: 19,324 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.60
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 18.9%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 40.9 seconds
After doublet removal: 19,321 cells
After MT filter: 15,966 cells
Normalised (UMI) – max value 13.56


In [10]:
study_name = 'Griffiths2021_iCell8'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 804 cells × 16,492 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 789 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.30
Detected doublet rate = 0.3%
Estimated detectable doublet fraction = 19.1%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.3%
Elapsed time: 1.0 seconds
After doublet removal: 787 cells
After MT filter: 787 cells
Normalised (UMI) – max value 11.00


In [11]:
study_name = 'Griffiths2021_10X'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 110,568 cells × 21,275 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 109,125 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.64
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 26.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 498.8 seconds
After doublet removal: 109,121 cells
After MT filter: 108,684 cells
Normalised (UMI) – max value 13.27


In [12]:
study_name = 'Gulati2020'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 1,902 cells × 42,132 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 1,856 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.41
Detected doublet rate = 0.3%
Estimated detectable doublet fraction = 8.9%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 3.0%
Elapsed time: 3.3 seconds
After doublet removal: 1,851 cells
After MT filter: 1,689 cells
Normalised (TPM) – max value 12.44


In [13]:
study_name = 'Karaayvas2018'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 1,534 cells × 21,785 genes
   cell_type: 422 NaN
   cell_type: 422 suspicious
Flagged 422 cells
After metadata drop: 1,112 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 996 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.34
Detected doublet rate = 0.5%
Estimated detectable doublet fraction = 8.5%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 5.9%
Elapsed time: 1.3 seconds
After doublet removal: 991 cells
After MT filter: 991 cells
Normalised (TPM) – max value 13.73


In [14]:
study_name = 'Kim2018'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 2,472 cells × 12,472 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 2,472 cells
Preprocessing...


/home/ubuntu/.local/lib/python3.12/site-packages/scrublet/helper_functions.py:239: RuntimeWarning: invalid value encountered in log
  gLog = lambda input: np.log(input[1] * np.exp(-input[0]) + input[2])
/home/ubuntu/.local/lib/python3.12/site-packages/scrublet/helper_functions.py:252: RuntimeWarning: invalid value encountered in sqrt
  CV_input = np.sqrt(b);


Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.43
Detected doublet rate = 0.6%
Estimated detectable doublet fraction = 29.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 2.2%
Elapsed time: 2.0 seconds
After doublet removal: 2,456 cells
After MT filter: 2,456 cells
Normalised (TPM) – max value 2.85


In [15]:
study_name = 'Pal2021_Group1'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 62,887 cells × 33,538 genes
   cell_type: 42860 NaN
   cell_type: 42860 suspicious
Flagged 42860 cells
After metadata drop: 20,027 cells
   Added cancer_type column. Unique values: ['Normal' 'Breast Cancer']
After gene filter: 19,980 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.53
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 34.5%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.2%
Elapsed time: 33.9 seconds
After doublet removal: 19,969 cells
After MT filter: 19,803 cells
Normalised (UMI) – max value 13.18


In [16]:
study_name = 'Pal2021_Group2'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 44,225 cells × 33,538 genes
   cell_type: 40443 NaN
   cell_type: 40443 suspicious
Flagged 40443 cells
After metadata drop: 3,782 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 3,780 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.37
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 34.8%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.6%
Elapsed time: 3.7 seconds
After doublet removal: 3,772 cells
After MT filter: 3,609 cells
Normalised (UMI) – max value 13.26


In [17]:
study_name = 'Pal2021_Group3'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 64,201 cells × 33,538 genes
   cell_type: 53377 NaN
   cell_type: 53377 suspicious
Flagged 53377 cells
After metadata drop: 10,824 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 10,712 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.50
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 31.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.5%
Elapsed time: 16.5 seconds
After doublet removal: 10,694 cells
After MT filter: 8,841 cells
Normalised (UMI) – max value 13.37


In [18]:
study_name = 'Pal2021_Group4'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 65,129 cells × 33,538 genes
   cell_type: 43369 NaN
   cell_type: 43369 suspicious
Flagged 43369 cells
After metadata drop: 21,760 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 21,337 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.61
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 19.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 43.6 seconds
After doublet removal: 21,331 cells
After MT filter: 19,238 cells
Normalised (UMI) – max value 13.57


In [19]:
study_name = 'Pal2021_Group5'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 68,782 cells × 33,538 genes
   cell_type: 32008 NaN
   cell_type: 32008 suspicious
Flagged 32008 cells
After metadata drop: 36,774 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 36,578 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.49
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 37.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.2%
Elapsed time: 85.9 seconds
After doublet removal: 36,556 cells
After MT filter: 34,916 cells
Normalised (UMI) – max value 13.56


In [20]:
study_name = 'Qian2020'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 16,537 cells × 22,276 genes
   cell_type: 865 NaN
   cell_type: 865 suspicious
Flagged 865 cells
After metadata drop: 15,672 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 15,672 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.57
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 21.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.2%
Elapsed time: 29.7 seconds
After doublet removal: 15,664 cells
After MT filter: 15,664 cells
Normalised (UMI) – max value 13.50


In [21]:
study_name = 'Savas2018'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 6,311 cells × 24,410 genes
   cell_type: 187 NaN
   cell_type: 187 suspicious
Flagged 187 cells
After metadata drop: 6,124 cells
   Added cancer_type column. Unique values: ['Breast Cancer']
After gene filter: 6,124 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.57
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 5.8 seconds
After doublet removal: 6,124 cells
After MT filter: 6,119 cells
Normalised (UMI) – max value 12.81


In [24]:
study_name = 'Wu2021'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# ensure cancer_type exists
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Breast'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# all metadata columns and the index to plain strings
for col in adata.obs.columns:
    try:
        adata.obs[col] = adata.obs[col].astype(str)
    except Exception:
        pass
adata.obs_names = adata.obs_names.astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

/tmp/ipykernel_488/829065860.py:11: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  cells = pd.read_csv(paths["cells"], index_col=0)


Loaded: 100,064 cells × 29,733 genes
   cancer_type: 10198 NaN
   cancer_type: 10198 suspicious
Flagged 10198 cells
After metadata drop: 89,866 cells
After gene filter: 88,608 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.83
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 1.9%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 339.7 seconds
After doublet removal: 88,607 cells
After MT filter: 85,526 cells
Normalised (UMI) – max value 13.72


In [25]:
INPUT_PATTERN = "*.h5ad"   

files = sorted(glob.glob(INPUT_PATTERN))

for f in files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0] if 'study' in adata.obs.columns else f
    print(f"{study_name}  –  {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    
    for col in sorted(adata.obs.columns):
        if pd.api.types.is_numeric_dtype(adata.obs[col]):
            continue
        
        vals = adata.obs[col].dropna().unique()
        n_unique = len(vals)
        
        if n_unique <= 25:
            val_str = ', '.join(str(v) for v in sorted(vals))
        else:
            val_str = f"{n_unique} unique values – first 15: " + ', '.join(str(v) for v in sorted(vals)[:15])
        
        print(f" {col}: {val_str}")
    
    print()

Azizi2018_10x  –  27,933 cells, 12,908 genes
 cancer_type: Breast Cancer
 cell_subtype: CD4 T cell, CD8 T cell, T-reg
 cell_type: T_cell
 cluster: 34 unique values – first 15: 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21, 22
 complexity: 3090 unique values – first 15: 1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014
 doublet_score: 217 unique values – first 15: 0.003380347359832149, 0.003914126438144941, 0.004050512270669527, 0.004188105779585977, 0.004326923076923077, 0.004466980562598093, 0.004608294930875577, 0.004750883177000853, 0.004894762604013705, 0.005039950829748003, 0.005186465794023217, 0.005334325766033991, 0.005483549351944167, 0.005634155502691875, 0.00578616352201258
 log1p_n_genes_by_counts: 3090 unique values – first 15: 5.66988092298052, 6.293419278846481, 6.311734809152915, 6.335054251498059, 6.345636360828596, 6.366470447731438, 6.373319789577012, 6.3818160174060985, 6.395261598115449, 6.401917196727186, 6.41509695917

In [26]:
import scanpy as sc
import pandas as pd
import numpy as np
import glob
import os
import gc
import anndata as ad
ad.settings.allow_write_nullable_strings = True

INPUT_PATTERN = "*.h5ad"          
OUTPUT_FILE = "breast.h5ad"

# All breast studies except Bassez2021
VALID_PREFIXES = [
    'Azizi2018_10x', 'Azizi2018_InDrop', 'Chung2017',
    'Gao2021_Breast', 'Gao2021_Breast_and_thyroid',
    'Griffiths2021_10X', 'Griffiths2021_iCell8',
    'Gulati2020', 'Karaayvas2018', 'Kim2018',
    'Pal2021_Group1', 'Pal2021_Group2', 'Pal2021_Group3',
    'Pal2021_Group4', 'Pal2021_Group5',
    'Qian2020', 'Savas2018', 'Wu2021'
]

all_files = sorted(glob.glob(INPUT_PATTERN))
study_files = [f for f in all_files if any(f.startswith(p) for p in VALID_PREFIXES)]
print(f"Found {len(study_files)} study files (expected 18)\n")

def label_malignant(adata, study_name):
    if 'cell_type' in adata.obs.columns:
        ct = adata.obs['cell_type'].astype(str).str.lower().str.strip()
        if ct.str.contains('malignant', na=False).any():
            return ct.str.contains('malignant', na=False), "cell_type='Malignant'"
    # Fallback: all non‑malignant
    return pd.Series(False, index=adata.obs.index), "all non‑malignant"

temp_files = []
total_malig = 0
total_non = 0

for f in study_files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0]   # from the per‑study script
    print(f"\nProcessing {study_name} ({adata.n_obs} cells) …")
    
    # --- Cancer type column ---
    if 'cancer_type' not in adata.obs.columns or adata.obs['cancer_type'].isna().all():
        adata.obs['cancer_type'] = 'Breast Cancer'
    else:
        adata.obs['cancer_type'] = adata.obs['cancer_type'].astype(str).fillna('Breast Cancer')
    
    # --- Malignancy labels ---
    malignant_mask, strategy = label_malignant(adata, study_name)
    adata.obs['is_malignant'] = malignant_mask.astype(int)
    
    malig_count = malignant_mask.sum()
    non_count = (~malignant_mask).sum()
    total_malig += malig_count
    total_non += non_count
    print(f"   malignant: {malig_count:>6,}   non‑malignant: {non_count:>7,}   ({strategy})")
    
    
    essential = ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']
    keep = [c for c in essential if c in adata.obs.columns]
    adata.obs = adata.obs[keep]
    
    
    for col in adata.obs.columns:
        try:
            adata.obs[col] = adata.obs[col].astype(str)
        except:
            pass
    adata.obs_names = adata.obs_names.astype(str)
    
  # temp filse
    temp_file = f"__temp_{study_name}.h5ad"
    adata.write_h5ad(temp_file)
    temp_files.append(temp_file)

print(f"TOTAL: {total_malig+total_non:>7,} cells  →  malignant: {total_malig:>6,}   non‑malignant: {total_non:>7,}")


print(f"\nMerging {len(temp_files)} studies incrementally …")

adata_all = sc.read_h5ad(temp_files[0])
print(f"   Starting with: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

for i, tf in enumerate(temp_files[1:], 1):
    print(f"   Adding study {i} …")
    ad = sc.read_h5ad(tf)
    
    adata_all = adata_all.concatenate(
        ad,
        batch_key='study',
        join='inner',
        index_unique='-'
    )
    # free the just‑added study
    del ad
    gc.collect()
    print(f"      Combined now: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

print(f"\nFinal merged object: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")


study_names_original = []
for tf in temp_files:
    ad = sc.read_h5ad(tf)
    study_names_original.append(ad.obs['study'].iloc[0])
study_map = {str(i): name for i, name in enumerate(study_names_original)}
adata_all.obs['study'] = adata_all.obs['study'].astype(str).map(study_map)
print(f"Study names restored: {sorted(adata_all.obs['study'].unique())}")

print("\nCancer types:", sorted(adata_all.obs['cancer_type'].astype(str).unique()))
print("Malignant distribution:")
print(adata_all.obs['is_malignant'].value_counts())

for col in adata_all.obs.columns:
    n_nan = adata_all.obs[col].isna().sum()
    if n_nan > 0:
        print(f"{col}: {n_nan} NaN – dropping")
        adata_all = adata_all[~adata_all.obs[col].isna(), :].copy()

for col in adata_all.obs.columns:
    try:
        adata_all.obs[col] = adata_all.obs[col].astype(str)
    except:
        pass
adata_all.obs_names = adata_all.obs_names.astype(str)

adata_all.write_h5ad(OUTPUT_FILE)

# Remove temporary files
for tf in temp_files:
    os.remove(tf)

Found 18 study files (expected 18)


Processing Azizi2018_10x (27933 cells) …
   malignant:      0   non‑malignant:  27,933   (all non‑malignant)

Processing Azizi2018_InDrop (44555 cells) …
   malignant:      0   non‑malignant:  44,555   (all non‑malignant)

Processing Chung2017 (155 cells) …
   malignant:     26   non‑malignant:     129   (cell_type='Malignant')

Processing Gao2021_Breast (3214 cells) …
   malignant:  2,076   non‑malignant:   1,138   (cell_type='Malignant')

Processing Gao2021_Breast_and_thyroid (15966 cells) …
   malignant:  5,665   non‑malignant:  10,301   (cell_type='Malignant')

Processing Griffiths2021_10X (108684 cells) …
   malignant: 108,684   non‑malignant:       0   (cell_type='Malignant')

Processing Griffiths2021_iCell8 (787 cells) …
   malignant:    787   non‑malignant:       0   (cell_type='Malignant')

Processing Gulati2020 (1689 cells) …
   malignant:    741   non‑malignant:     948   (cell_type='Malignant')

Processing Karaayvas2018 (991 cells) …
   

/tmp/ipykernel_488/1116973703.py:91: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(


      Combined now: 72,488 cells × 11,253 genes
   Adding study 2 …


/tmp/ipykernel_488/1116973703.py:91: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(


      Combined now: 72,643 cells × 10,958 genes
   Adding study 3 …


...

      Combined now: 308,501 cells × 6,702 genes
   Adding study 16 …


/tmp/ipykernel_488/1116973703.py:91: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(


      Combined now: 314,620 cells × 6,699 genes
   Adding study 17 …


/tmp/ipykernel_488/1116973703.py:91: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(


      Combined now: 400,146 cells × 6,699 genes

Final merged object: 400,146 cells × 6,699 genes
Study names restored: ['Azizi2018_10x', 'Azizi2018_InDrop']

Cancer types: ['Breast Cancer', 'IDC', 'ILC', 'MBC', 'Normal', 'Premalignant', 'Thyroid Cancer']
Malignant distribution:
is_malignant
0    205270
1    194876
Name: count, dtype: int64
